In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 6: TurboQuant — Rotating Your Way to Better Cache Compression

*Part of the Vizuara series on KV Cache Optimization*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Imagine you are running a language model serving thousands of users. Each user has a long conversation context — perhaps a 4,000-token document. As we learned in earlier sections, the KV cache stores all those keys and values in GPU memory so we do not recompute attention from scratch on every token. That is the hero of inference efficiency.

But there is a villain lurking inside the hero. Storing all those cached keys and values takes up enormous amounts of VRAM. For a typical 7B-parameter model with 32 attention heads and a 4,000-token context, the KV cache alone can consume **gigabytes** of memory. This forces us to either serve fewer users simultaneously (lower throughput) or use shorter contexts (lower quality). Neither is acceptable.

**The solution that top AI labs use today is KV cache quantization** — compressing those stored tensors from 16-bit floats down to 4 bits per value. That is a 4× memory reduction. But naive quantization destroys model quality. This notebook is about *why* naive quantization fails and how a surprisingly elegant mathematical trick — **random rotation** — fixes it.

By the end of this notebook, you will be able to:
- Understand exactly why some vectors are hard to quantize
- See how rotation makes vectors quantization-friendly
- Implement a full TurboQuant-style pipeline with rotation + quantization
- Measure and visualize the quality improvement

Let us see a teaser of where we are headed:

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
torch.manual_seed(42)
np.random.seed(42)

# ── Teaser: what does rotation do for quantization? ──────────────────────────
# We will build up to all of this — for now, just appreciate the picture.

def hadamard_matrix(n):
    """Build a Hadamard matrix of size n (n must be power of 2)."""
    if n == 1:
        return torch.ones(1, 1)
    H_half = hadamard_matrix(n // 2)
    top    = torch.cat([H_half,  H_half], dim=1)
    bottom = torch.cat([H_half, -H_half], dim=1)
    return torch.cat([top, bottom], dim=0) / (2 ** 0.5)

def quantize_4bit(x):
    """Uniform 4-bit quantization (16 levels) over a vector."""
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / 15.0          # 2^4 - 1 = 15 levels
    x_q   = torch.round((x - x_min) / scale)
    x_dq  = x_q * scale + x_min             # dequantize back
    return x_dq

# Simulate a key vector with the "outlier" problem we'll study in depth
dim   = 64
k_vec = torch.randn(dim) * 0.3
k_vec[7]  =  8.5   # a large outlier — very common in real transformers!
k_vec[31] = -7.2

# Apply rotation then quantize vs. just quantize
H    = hadamard_matrix(dim)
k_rot = (H @ k_vec)                     # rotate
k_rot_q = quantize_4bit(k_rot)          # quantize rotated
k_rot_dq = (H.T @ k_rot_q)             # rotate back

k_naive_dq = quantize_4bit(k_vec)       # naive: quantize directly

err_naive  = ((k_vec - k_naive_dq)**2).mean().item()
err_turbo  = ((k_vec - k_rot_dq)**2).mean().item()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(range(dim), k_vec.numpy(), color='steelblue', alpha=0.8)
axes[0].set_title('Original Key Vector\n(notice the outliers!)', fontsize=12)
axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Value')

axes[1].bar(range(dim), (k_vec - k_naive_dq).numpy(), color='crimson', alpha=0.8)
axes[1].set_title(f'Naive Quantization Error\nMSE = {err_naive:.4f}', fontsize=12)
axes[1].set_xlabel('Dimension')

axes[2].bar(range(dim), (k_vec - k_rot_dq).numpy(), color='seagreen', alpha=0.8)
axes[2].set_title(f'TurboQuant Error\nMSE = {err_turbo:.4f}', fontsize=12)
axes[2].set_xlabel('Dimension')

plt.suptitle('🚀 TurboQuant: Rotation Dramatically Reduces Quantization Error',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('teaser.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"\n✨ TurboQuant is {err_naive/err_turbo:.1f}× better than naive quantization!")

## 2. Building Intuition

Let us think carefully about what quantization actually does — and *why* it sometimes fails badly.

### 🎯 Quantization as a Ruler

When you quantize a vector, you are replacing each number with the nearest value on a **fixed grid**. With 4-bit quantization, that grid has exactly **16 levels** (since $2^4 = 16$). Those 16 levels must span the *entire* range of the vector — from its minimum value to its maximum.

Think of it like a ruler. If your ruler goes from 0 cm to 30 cm with 16 tick marks, those tick marks are spaced about 2 cm apart. Any measurement gets rounded to the nearest tick mark. That rounding error — at most 1 cm — is your quantization noise.

Now here is the key question: **what happens when one dimension of your vector is 100 cm?**

Suddenly your ruler must span 0 to 100 cm. Those same 16 tick marks now cover the entire 100 cm range, spacing them about 6.7 cm apart. Everything that was packed neatly between 0 and 30 cm now gets rounded to coarse 6.7 cm intervals. **One outlier ruins the quantization grid for everyone.**

### 🦁 The Outlier Problem in Real Transformers

This is not hypothetical. Neural networks trained on real data develop **outlier dimensions** — a handful of dimensions in their key and value vectors that regularly take on values 10× or 100× larger than the rest. This was documented extensively in the LLM.int8() paper by Tim Dettmers and colleagues.

Why do these outliers arise? Attention heads learn to use certain dimensions as "routing signals" — strong activations that determine which tokens are attended to. These signals need to be large to stand out. The network learns to concentrate information into a few loud dimensions, leaving the rest quiet.

The result: a 64-dimensional key vector where 62 dimensions are between -1 and +1, and 2 dimensions are at ±8. When you try to quantize this with 16 levels spanning -8 to +8, each level is 1.07 units wide. All those quiet dimensions between -1 and +1 share only about 2 of those 16 levels — effectively becoming 1-bit representation for most of the vector!

### 🌀 The Rotation Idea

Here is the beautiful insight: **quantization error is a property of the vector, not of the information it contains.** If we could somehow *spread* the information evenly across all dimensions before quantizing, every dimension would have similar magnitude — and our 16-level grid would fit them all well.

A rotation does exactly this. A rotation matrix $R$ transforms a vector $\mathbf{v}$ into $R\mathbf{v}$ in a way that **perfectly preserves all information** (it is an invertible linear transformation with $R^{-1} = R^T$). But it redistributes the "energy" across dimensions.

Think of stirring a glass of water with a drop of food coloring. Before stirring: the color is concentrated in one spot (outlier!). After stirring: the color is spread uniformly (no outliers — perfect for quantization). The total amount of color — the total information — is unchanged.

The trick: rotate → quantize → rotate back. The rotation mixes the outlier energy across all dimensions, quantization works well on the nicely spread result, and rotating back recovers the original space.

### 🤔 Think About This

Before we dive into the mathematics: why does a rotation preserve information perfectly while quantization does not? What property of a rotation matrix makes the rotate→quantize→rotate-back scheme lossless in one direction but helpful in the other?

*Hint: think about what the matrix $R^T R$ equals for a rotation matrix...*

## 3. The Mathematics

Let us now formalize everything we intuited above.

### 3.1 The Quantization Operation

Let $\mathbf{v} \in \mathbb{R}^d$ be a vector we wish to quantize to $b$ bits. Uniform quantization works as follows:

$$\hat{\mathbf{v}} = s \cdot \text{round}\!\left(\frac{\mathbf{v} - v_{\min}}{s}\right) + v_{\min}$$

where the scale factor is:

$$s = \frac{v_{\max} - v_{\min}}{2^b - 1}$$

This equation says: take each element of $\mathbf{v}$, subtract the minimum (so everything starts at zero), divide by the step size $s$ (so values are on a 0-to-15 integer grid for 4-bit), round to the nearest integer, then multiply by $s$ and add back the minimum to recover the original scale. Computationally, this means every value is rounded to the nearest multiple of $s$ above $v_{\min}$.

The **quantization error** for a single vector is:

$$\mathcal{E}(\mathbf{v}) = \|\mathbf{v} - \hat{\mathbf{v}}\|_2^2 = \sum_{i=1}^{d} (v_i - \hat{v}_i)^2$$

The worst-case error per element is bounded by $(s/2)^2$. So:

$$\mathcal{E}(\mathbf{v}) \leq d \cdot \left(\frac{s}{2}\right)^2 = d \cdot \left(\frac{v_{\max} - v_{\min}}{2(2^b - 1)}\right)^2$$

This equation says: the error grows with the *square of the range* of the vector. **This is why outliers are so catastrophically bad** — one outlier doubles the range, and the error goes up by a factor of four.

### 3.2 The Rotation Trick

Now let $R \in \mathbb{R}^{d \times d}$ be an orthogonal matrix satisfying:

$$R^T R = R R^T = I$$

This is the key orthogonality property. It means $R^{-1} = R^T$ — inverting a rotation is free (just transpose!).

The TurboQuant pipeline is:

$$\tilde{\mathbf{v}} = R\mathbf{v} \quad \xrightarrow{\text{quantize}} \quad \hat{\tilde{\mathbf{v}}} \quad \xrightarrow{\text{rotate back}} \quad \hat{\mathbf{v}} = R^T \hat{\tilde{\mathbf{v}}}$$

The reconstruction error in the *original* space is:

$$\|\mathbf{v} - \hat{\mathbf{v}}\|_2^2 = \|R\mathbf{v} - R\hat{\mathbf{v}}\|_2^2 = \|\tilde{\mathbf{v}} - \hat{\tilde{\mathbf{v}}}\|_2^2$$

This equation says: rotating is an **isometry** — it preserves distances. So the error in the original space equals exactly the error in the rotated space. We have not made quantization magically more accurate in terms of absolute error — we have made the *distribution* of that error more uniform, which in practice gives us much lower error when the original vector has outliers.

### 3.3 Why the Hadamard Matrix?

The ideal rotation would perfectly equalize all dimensions. The **Hadamard matrix** of order $d$ (for $d$ a power of 2) does something close to this:

$$H_d = \frac{1}{\sqrt{d}} \begin{pmatrix} H_{d/2} & H_{d/2} \\ H_{d/2} & -H_{d/2} \end{pmatrix}, \quad H_1 = [1]$$

Each row of $H_d$ is a $\pm 1/\sqrt{d}$ sequence (a Walsh function). When $H_d$ multiplies a vector $\mathbf{v}$, the result is a **discrete Walsh-Hadamard Transform**. Each output dimension is a sum of all input dimensions (with various signs), divided by $\sqrt{d}$.

This means: if $\mathbf{v}$ has one huge outlier at dimension $k$, after applying $H_d$ that outlier's energy is **spread equally** across all $d$ output dimensions, each receiving $\pm \text{outlier}/\sqrt{d}$. The maximum value in the rotated vector scales as $O(\text{outlier}/\sqrt{d})$ rather than $O(\text{outlier})$.

Better still, the Hadamard transform can be computed in $O(d \log d)$ time using the **Fast Walsh-Hadamard Transform (FWHT)** — analogous to how FFT speeds up the Fourier transform. No matrix-vector multiplication needed. That is why it is used in practice.

## 4. Let's Build It — Component by Component

### 4.1 Uniform Quantization from Scratch

Let us implement quantization carefully, understanding every step.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(42)
np.random.seed(42)

def quantize_uniform(x: torch.Tensor, bits: int = 4) -> tuple:
    """
    Uniform quantization of a 1D tensor.

    Parameters
    ----------
    x    : tensor to quantize
    bits : number of bits (4 → 16 levels)

    Returns
    -------
    x_dq    : dequantized tensor (same shape as x, float)
    x_min   : minimum value used for scaling
    scale   : step size s
    """
    levels = 2**bits - 1          # e.g. 15 for 4-bit
    x_min  = x.min()
    x_max  = x.max()
    scale  = (x_max - x_min) / levels

    # Quantize: map to integer grid [0, levels]
    x_int = torch.round((x - x_min) / scale).clamp(0, levels)

    # Dequantize: map back to original scale
    x_dq  = x_int * scale + x_min

    return x_dq, x_min, scale

# ── Let us see this in action on a clean vector ──────────────────────────────
clean_vec = torch.randn(32) * 0.5      # small, well-behaved values
dq_clean, _, _ = quantize_uniform(clean_vec, bits=4)

print("=== Clean vector quantization ===")
print(f"  Range: [{clean_vec.min():.3f}, {clean_vec.max():.3f}]")
print(f"  Range width: {(clean_vec.max() - clean_vec.min()):.3f}")
print(f"  Step size (scale): {(clean_vec.max()-clean_vec.min())/15:.4f}")
print(f"  MSE: {((clean_vec - dq_clean)**2).mean():.6f}")

# Now add an outlier and see what happens
outlier_vec = clean_vec.clone()
outlier_vec[5] = 9.0     # one big outlier
dq_outlier, _, s_out = quantize_uniform(outlier_vec, bits=4)

print("\n=== Same vector, but with one outlier at index 5 ===")
print(f"  Range: [{outlier_vec.min():.3f}, {outlier_vec.max():.3f}]")
print(f"  Range width: {(outlier_vec.max() - outlier_vec.min()):.3f}")
print(f"  Step size (scale): {s_out:.4f}")
print(f"  MSE: {((outlier_vec - dq_outlier)**2).mean():.6f}")
print(f"\n  ⚠️  The step size grew by ~{s_out/((clean_vec.max()-clean_vec.min())/15):.1f}×!")
print(f"  ⚠️  MSE worsened by ~{((outlier_vec - dq_outlier)**2).mean() / ((clean_vec - dq_clean)**2).mean():.0f}×!")

In [ ]:
# 📊 Visualization: how the grid changes with outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# --- Top row: value distributions ---
axes[0, 0].hist(clean_vec.numpy(), bins=20, color='steelblue', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('Clean Vector Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(outlier_vec.numpy(), bins=20, color='crimson', alpha=0.8, edgecolor='white')
axes[0, 1].set_title('Outlier Vector Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Value')

# --- Bottom row: quantization grid visualization ---
for ax, vec, dq, title, color in [
    (axes[1, 0], clean_vec,   dq_clean,   'Clean Vector', 'steelblue'),
    (axes[1, 1], outlier_vec, dq_outlier, 'Outlier Vector', 'crimson')
]:
    v_min, v_max = vec.min().item(), vec.max().item()
    scale = (v_max - v_min) / 15
    grid_lines = [v_min + i * scale for i in range(16)]

    ax.scatter(range(len(vec)), vec.numpy(), color=color, label='Original', zorder=5, s=40)
    ax.scatter(range(len(vec)), dq.numpy(), color='orange', marker='x',
               label='Quantized', zorder=6, s=60, linewidths=2)
    for g in grid_lines:
        ax.axhline(g, color='gray', alpha=0.3, linewidth=0.8, linestyle='--')
    ax.set_title(f'{title}\n(grid lines = quantization levels)', fontweight='bold')
    ax.set_xlabel('Dimension index')
    ax.set_ylabel('Value')
    ax.legend()

plt.suptitle('🔍 The Outlier Problem: One Bad Value Ruins the Grid',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_problem.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 The Hadamard Matrix and Fast Walsh-Hadamard Transform

Now let us implement the rotation we will use. We will build the Hadamard matrix recursively and also implement the fast transform.

In [ ]:
def hadamard_matrix(n: int) -> torch.Tensor:
    """
    Build the normalized Hadamard matrix H_n (n must be a power of 2).
    H_n is orthogonal: H_n @ H_n.T = I
    Each element has magnitude 1/sqrt(n).
    """
    assert (n & (n - 1)) == 0, "n must be a power of 2"
    if n == 1:
        return torch.ones(1, 1, dtype=torch.float32)
    H_half = hadamard_matrix(n // 2)
    # Block construction: [[H, H], [H, -H]] / sqrt(2)
    top    = torch.cat([H_half,  H_half], dim=1)
    bottom = torch.cat([H_half, -H_half], dim=1)
    return torch.cat([top, bottom], dim=0) / (2 ** 0.5)

def fast_hadamard_transform(x: torch.Tensor) -> torch.Tensor:
    """
    Fast Walsh-Hadamard Transform (FWHT) — O(n log n) instead of O(n^2).
    Equivalent to H_n @ x but without forming the full matrix.
    """
    n = x.shape[-1]
    assert (n & (n - 1)) == 0, "Last dimension must be power of 2"
    result = x.clone().float()
    step = 1
    while step < n:
        for i in range(0, n, step * 2):
            # Butterfly operation
            a = result[..., i : i + step].clone()
            b = result[..., i + step : i + 2 * step].clone()
            result[..., i : i + step]         = a + b
            result[..., i + step : i + 2 * step] = a - b
        step *= 2
    return result / (n ** 0.5)

# ── Verify correctness ────────────────────────────────────────────────────────
d   = 16
H   = hadamard_matrix(d)
v   = torch.randn(d)

h_slow = H @ v                         # matrix multiply — O(n^2)
h_fast = fast_hadamard_transform(v)    # butterfly algorithm — O(n log n)

print("=== Correctness check ===")
print(f"  Max difference between slow and fast: {(h_slow - h_fast).abs().max():.2e}")
print(f"  ✅ Both methods agree!" if (h_slow - h_fast).abs().max() < 1e-5 else "  ❌ Mismatch!")

print("\n=== Orthogonality check: H.T @ H should = I ===")
err = (H.T @ H - torch.eye(d)).abs().max()
print(f"  Max deviation from identity: {err:.2e}")
print(f"  ✅ H is orthogonal!" if err < 1e-5 else "  ❌ Not orthogonal!")

print("\n=== Energy preservation check ===")
v_norm  = (v**2).sum().item()
hv_norm = (h_fast**2).sum().item()
print(f"  ||v||²  = {v_norm:.4f}")
print(f"  ||Hv||² = {hv_norm:.4f}")
print(f"  ✅ Energy preserved!" if abs(v_norm - hv_norm) < 1e-4 else "  ❌ Energy changed!")

In [ ]:
# 📊 Visualize what Hadamard rotation does to a vector with outliers
d = 64
H = hadamard_matrix(d)

v_original = torch.randn(d) * 0.3
v_original[7]  =  8.5    # outliers
v_original[31] = -7.2

v_rotated = H @ v_original

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(range(d), v_original.numpy(), color='steelblue', alpha=0.85)
axes[0].set_title(f'Original Vector\nmax|v| = {v_original.abs().max():.2f}', fontweight='bold')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Value')
axes[0].axhline(0, color='black', linewidth=0.5)

axes[1].bar(range(d), v_rotated.numpy(), color='mediumpurple', alpha=0.85)
axes[1].set_title(f'After Hadamard Rotation\nmax|Hv| = {v_rotated.abs().max():.2f}', fontweight='bold')
axes[1].set_xlabel('Dimension')

# Show energy distribution (sorted absolute values)
orig_sorted = v_original.abs().sort(descending=True).values.numpy()
rot_sorted  = v_rotated.abs().sort(descending=True).values.numpy()
axes[2].plot(orig_sorted,  color='steelblue',    label='Original',  linewidth=2)
axes[2].plot(rot_sorted,   color='mediumpurple', label='Rotated',   linewidth=2)
axes[2].set_title('Energy Distribution\n(sorted absolute values)', fontweight='bold')
axes[2].set_xlabel('Rank (0 = largest)')
axes[2].set_ylabel('|value|')
axes[2].legend()
axes[2].fill_between(range(d), rot_sorted, alpha=0.2, color='mediumpurple')

plt.suptitle('🌀 Hadamard Rotation Spreads Outlier Energy Across All Dimensions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hadamard_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Original: max|v| = {v_original.abs().max():.3f},  "
      f"std|v| = {v_original.abs().std():.3f}")
print(f"Rotated:  max|Hv| = {v_rotated.abs().max():.3f},  "
      f"std|Hv| = {v_rotated.abs().std():.3f}")
print(f"\n✅ After rotation, the max value dropped from "
      f"{v_original.abs().max():.1f} to {v_rotated.abs().max():.2f}")

### 4.3 The Full TurboQuant Pipeline

Now let us assemble the complete pipeline: rotate → quantize → store. At decode time: load → dequantize → rotate back.

In [ ]:
class TurboQuant:
    """
    TurboQuant: Rotation-based KV cache quantization.

    Pipeline:
        compress:   v  →  H @ v  →  quantize(H @ v)  →  (x_int, scale, x_min)
        decompress: (x_int, scale, x_min)  →  dequantize  →  H.T @ result  →  v̂
    """

    def __init__(self, dim: int, bits: int = 4):
        assert (dim & (dim - 1)) == 0, "dim must be a power of 2"
        self.dim  = dim
        self.bits = bits
        self.levels = 2**bits - 1
        self.H    = hadamard_matrix(dim)

    def compress(self, v: torch.Tensor) -> dict:
        """
        Compress a vector using rotation + quantization.

        Parameters
        ----------
        v : shape (dim,) or (..., dim) — the key or value vector

        Returns
        -------
        A dict with keys: 'x_int' (uint8), 'scale' (float), 'x_min' (float)
        """
        # Step 1: rotate
        v_rot = v @ self.H.T   # works for batched input too: (..., dim) @ (dim, dim)

        # Step 2: compute scale per-token (last dimension is feature dim)
        v_min   = v_rot.min(dim=-1, keepdim=True).values
        v_max   = v_rot.max(dim=-1, keepdim=True).values
        scale   = (v_max - v_min) / self.levels

        # Step 3: quantize to integer
        x_int = torch.round((v_rot - v_min) / scale.clamp(min=1e-8))
        x_int = x_int.clamp(0, self.levels).to(torch.uint8)

        return {'x_int': x_int, 'scale': scale, 'x_min': v_min}

    def decompress(self, compressed: dict) -> torch.Tensor:
        """
        Decompress: dequantize then rotate back.
        """
        x_int   = compressed['x_int'].float()
        scale   = compressed['scale']
        x_min   = compressed['x_min']

        # Step 1: dequantize
        v_rot_dq = x_int * scale + x_min

        # Step 2: rotate back (H is orthogonal, so H^{-1} = H.T)
        v_hat = v_rot_dq @ self.H   # (..., dim) @ (dim, dim)

        return v_hat

# ── Quick sanity check ────────────────────────────────────────────────────────
dim = 64
tq  = TurboQuant(dim=dim, bits=4)

# A batch of 10 key vectors, some with outliers
keys = torch.randn(10, dim) * 0.4
keys[2, 15] =  9.3    # outlier in token 2
keys[7, 40] = -8.1    # outlier in token 7

compressed = tq.compress(keys)
keys_hat   = tq.decompress(compressed)

print(f"Original shape:     {keys.shape}")
print(f"Compressed x_int:   {compressed['x_int'].shape}, dtype={compressed['x_int'].dtype}")
print(f"Reconstructed shape:{keys_hat.shape}")
print()
print(f"Memory before: {keys.numel() * 2} bytes  (float16)")
print(f"Memory after:  {compressed['x_int'].numel() // 2} bytes  (packed 4-bit, 2 values/byte)")
print(f"Compression ratio: {(keys.numel() * 2) / (compressed['x_int'].numel() // 2):.1f}×")
print()
mse_turbo = ((keys - keys_hat)**2).mean().item()
print(f"TurboQuant MSE: {mse_turbo:.6f}")

## 5. 🔧 Your Turn

Now it is your chance to build a crucial component. We want to compare TurboQuant against naive quantization (no rotation) and measure the benefit across different *outlier intensities*.

### TODO: Implement `naive_quantize` and `compute_reconstruction_stats`

In [ ]:
def naive_quantize(v: torch.Tensor, bits: int = 4) -> torch.Tensor:
    """
    Naive uniform quantization WITHOUT any rotation.

    Parameters
    ----------
    v    : input tensor of shape (..., dim)
    bits : number of quantization bits

    Returns
    -------
    v_hat : reconstructed tensor of same shape

    Hints
    -----
    # Step 1: compute levels = 2**bits - 1  (e.g. 15 for 4-bit)
    # Step 2: find v_min and v_max along the LAST dimension (keepdim=True)
    # Step 3: compute scale = (v_max - v_min) / levels
    # Step 4: quantize to integer: round((v - v_min) / scale), clamp to [0, levels]
    # Step 5: dequantize: x_int * scale + v_min
    """
    # ============ TODO ============
    # Step 1:
    levels = ???

    # Step 2:
    v_min = ???
    v_max = ???

    # Step 3:
    scale = ???

    # Step 4:
    x_int = ???

    # Step 5:
    v_hat = ???
    # ============ END TODO ========
    return v_hat


def compute_reconstruction_stats(v_original: torch.Tensor,
                                  v_hat: torch.Tensor) -> dict:
    """
    Compute reconstruction quality metrics.

    Parameters
    ----------
    v_original : original vectors, shape (N, dim)
    v_hat      : reconstructed vectors, shape (N, dim)

    Returns
    -------
    dict with keys: 'mse', 'cosine_sim', 'max_err'

    Hints
    -----
    # MSE: mean squared error over ALL elements
    # cosine_sim: for each of the N vectors, compute dot(v, v_hat) / (||v|| ||v_hat||)
    #             then return the MEAN cosine similarity across N vectors
    # max_err: maximum absolute error over ALL elements
    #
    # Useful: torch.nn.functional.cosine_similarity(a, b, dim=-1)
    """
    import torch.nn.functional as F
    # ============ TODO ============
    mse        = ???
    cosine_sim = ???
    max_err    = ???
    # ============ END TODO ========
    return {'mse': mse, 'cosine_sim': cosine_sim, 'max_err': max_err}

In [ ]:
# ✅ Verification cell — run this to check your implementations

# Test naive_quantize
test_v = torch.tensor([[1.0, 2.0, 3.0, 4.0,  # 4 clean values
                         0.5, 1.5, 2.5, 3.5]])  # shape (1, 8)
test_dq = naive_quantize(test_v, bits=4)
assert test_dq.shape == test_v.shape, "Output shape must match input"
assert test_dq.dtype == test_v.dtype or test_dq.dtype == torch.float32, "Must return float"
assert ((test_v - test_dq)**2).mean() < 0.1, "MSE on clean vector should be small"
print("✅ naive_quantize shape and dtype: OK")
print(f"   MSE on clean vector: {((test_v - test_dq)**2).mean():.5f}  (should be < 0.1)")

# Test with outlier
test_outlier = test_v.clone()
test_outlier[0, 0] = 50.0   # huge outlier
test_dq_out = naive_quantize(test_outlier, bits=4)
mse_with_outlier = ((test_outlier - test_dq_out)**2).mean()
assert mse_with_outlier > ((test_v - test_dq)**2).mean(), \
    "Outlier should make MSE worse"
print(f"✅ Outlier makes naive MSE worse: {mse_with_outlier:.3f}  >  "
      f"{((test_v - test_dq)**2).mean():.5f}")

# Test compute_reconstruction_stats
v1 = torch.randn(20, 32)
v1_hat = v1 + 0.01 * torch.randn_like(v1)
stats = compute_reconstruction_stats(v1, v1_hat)
assert 'mse' in stats and 'cosine_sim' in stats and 'max_err' in stats
assert 0 < stats['cosine_sim'] <= 1.0, "Cosine similarity should be between 0 and 1"
assert stats['mse'] > 0, "MSE should be positive"
print(f"✅ compute_reconstruction_stats: MSE={stats['mse']:.5f}, "
      f"cos_sim={stats['cosine_sim']:.4f}")
print("\n🎉 All checks passed! Let us move on.")

## 6. Putting It All Together

Let us now run a comprehensive experiment comparing naive quantization vs. TurboQuant across a range of outlier intensities and bit widths. This is exactly the kind of ablation study you would see in a real research paper.

In [ ]:
def run_ablation_study(dim=64, n_tokens=200, outlier_fraction=0.05, bits_list=[2, 4, 8]):
    """
    Compare naive vs. TurboQuant quantization across different bit widths.
    Injects outliers into a fraction of dimensions.
    """
    torch.manual_seed(42)
    n_outlier_dims = max(1, int(dim * outlier_fraction))

    results = {'naive': {}, 'turbo': {}}

    for bits in bits_list:
        tq  = TurboQuant(dim=dim, bits=bits)

        # Generate realistic key vectors (small background + a few large outliers)
        keys = torch.randn(n_tokens, dim) * 0.4
        outlier_dims = torch.randperm(dim)[:n_outlier_dims]
        outlier_magnitudes = torch.rand(n_tokens, n_outlier_dims) * 8 + 4  # 4 to 12
        outlier_signs      = (torch.randint(0, 2, (n_tokens, n_outlier_dims)) * 2 - 1).float()
        keys[:, outlier_dims] = outlier_magnitudes * outlier_signs

        # Naive quantization
        naive_hat  = naive_quantize(keys, bits=bits)
        stats_naive = compute_reconstruction_stats(keys, naive_hat)

        # TurboQuant
        compressed   = tq.compress(keys)
        turbo_hat    = tq.decompress(compressed)
        stats_turbo  = compute_reconstruction_stats(keys, turbo_hat)

        results['naive'][bits] = stats_naive
        results['turbo'][bits] = stats_turbo

        print(f"Bits={bits}: Naive MSE={stats_naive['mse']:.4f}, "
              f"TurboQuant MSE={stats_turbo['mse']:.4f}, "
              f"Speedup={stats_naive['mse']/stats_turbo['mse']:.1f}×")

    return results

print("Running ablation study...")
print("="*60)
results = run_ablation_study(dim=64, n_tokens=500, outlier_fraction=0.05,
                              bits_list=[2, 4, 8])

In [ ]:
# 📊 Publication-quality comparison plot
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

bits_list  = [2, 4, 8]
metrics    = ['mse', 'cosine_sim', 'max_err']
titles     = ['Mean Squared Error\n(lower is better)',
              'Cosine Similarity\n(higher is better)',
              'Max Absolute Error\n(lower is better)']
colors     = {'naive': '#e74c3c', 'turbo': '#2ecc71'}

for col, (metric, title) in enumerate(zip(metrics, titles)):
    ax = fig.add_subplot(gs[0, col])
    naive_vals = [results['naive'][b][metric] for b in bits_list]
    turbo_vals = [results['turbo'][b][metric] for b in bits_list]

    x  = np.arange(len(bits_list))
    bw = 0.35
    ax.bar(x - bw/2, naive_vals, bw, label='Naive Quant', color=colors['naive'], alpha=0.85)
    ax.bar(x + bw/2, turbo_vals, bw, label='TurboQuant',  color=colors['turbo'], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{b}-bit' for b in bits_list])
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('Bit Width')

# Bottom row: sweep over outlier intensity
ax_sweep = fig.add_subplot(gs[1, :])
outlier_intensities = np.linspace(0, 12, 30)
mse_naive_sweep = []
mse_turbo_sweep = []

dim, n_tokens, bits = 64, 200, 4
tq = TurboQuant(dim=dim, bits=bits)

for intensity in outlier_intensities:
    torch.manual_seed(42)
    keys = torch.randn(n_tokens, dim) * 0.3
    keys[:, [7, 31]] = intensity * (torch.randint(0, 2, (n_tokens, 2)) * 2 - 1).float()

    naive_hat = naive_quantize(keys, bits=bits)
    compressed = tq.compress(keys)
    turbo_hat = tq.decompress(compressed)

    mse_naive_sweep.append(((keys - naive_hat)**2).mean().item())
    mse_turbo_sweep.append(((keys - turbo_hat)**2).mean().item())

ax_sweep.plot(outlier_intensities, mse_naive_sweep, color=colors['naive'],
              linewidth=2.5, label='Naive Quantization', marker='o', markersize=4)
ax_sweep.plot(outlier_intensities, mse_turbo_sweep, color=colors['turbo'],
              linewidth=2.5, label='TurboQuant',          marker='s', markersize=4)
ax_sweep.fill_between(outlier_intensities, mse_naive_sweep, mse_turbo_sweep,
                       alpha=0.2, color='steelblue', label='Improvement gap')
ax_sweep.set_xlabel('Outlier Magnitude', fontsize=11)
ax_sweep.set_ylabel('MSE', fontsize=11)
ax_sweep.set_title('4-bit MSE vs. Outlier Magnitude: TurboQuant Stays Stable, Naive Explodes',
                    fontweight='bold', fontsize=11)
ax_sweep.legend(fontsize=10)
ax_sweep.set_xlim(0, 12)

fig.suptitle('📊 TurboQuant Ablation Study', fontsize=14, fontweight='bold')
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Simulating the Real KV Cache

Let us simulate what happens in a real transformer-style KV cache: we store keys over many generation steps, then retrieve and compute attention. This is where quantization error really matters — errors in keys distort the attention scores and corrupt the model's output.

In [ ]:
def compute_attention(queries: torch.Tensor,
                      keys:    torch.Tensor,
                      values:  torch.Tensor,
                      scale:   float = None) -> torch.Tensor:
    """Scaled dot-product attention."""
    d_k   = queries.shape[-1]
    scale = scale or (d_k ** -0.5)
    scores = (queries @ keys.T) * scale
    probs  = torch.softmax(scores, dim=-1)
    return probs @ values, probs


def kv_cache_simulation(seq_len: int = 128,
                         dim: int = 64,
                         n_heads: int = 4,
                         bits: int = 4,
                         outlier_prob: float = 0.05) -> dict:
    """
    Simulate a KV cache with compression.
    Generates a sequence of key-value pairs, stores them compressed,
    then measures attention quality when they are retrieved.
    """
    torch.manual_seed(42)
    tq = TurboQuant(dim=dim, bits=bits)

    # Simulate keys and values as they would be produced by a transformer
    keys   = torch.randn(seq_len, dim) * 0.4
    values = torch.randn(seq_len, dim) * 0.4

    # Inject realistic outliers
    for i in range(seq_len):
        if torch.rand(1).item() < outlier_prob:
            outlier_dim  = torch.randint(0, dim, (1,)).item()
            keys[i, outlier_dim] = (torch.rand(1).item() * 8 + 4) * (1 if torch.rand(1) > 0.5 else -1)

    # --- Reference: no compression ---
    query  = torch.randn(1, dim) * 0.4      # one new query token
    attn_ref, probs_ref = compute_attention(query, keys, values)

    # --- Naive quantization of the KV cache ---
    keys_naive   = naive_quantize(keys,   bits=bits)
    values_naive = naive_quantize(values, bits=bits)
    attn_naive, probs_naive = compute_attention(query, keys_naive, values_naive)

    # --- TurboQuant compression ---
    keys_turbo   = tq.decompress(tq.compress(keys))
    values_turbo = tq.decompress(tq.compress(values))
    attn_turbo, probs_turbo = compute_attention(query, keys_turbo, values_turbo)

    return {
        'keys': keys,
        'probs_ref':   probs_ref.squeeze(),
        'probs_naive': probs_naive.squeeze(),
        'probs_turbo': probs_turbo.squeeze(),
        'attn_ref':    attn_ref.squeeze(),
        'attn_naive':  attn_naive.squeeze(),
        'attn_turbo':  attn_turbo.squeeze(),
    }

print("Running KV cache simulation...")
sim = kv_cache_simulation(seq_len=128, dim=64, bits=4)

# Compare attention output quality
ref   = sim['attn_ref']
naive = sim['attn_naive']
turbo = sim['attn_turbo']

print(f"\nAttention output MSE:")
print(f"  Naive quantization: {((ref - naive)**2).mean():.5f}")
print(f"  TurboQuant:         {((ref - turbo)**2).mean():.5f}")
print(f"  Improvement:        {((ref - naive)**2).mean() / ((ref - turbo)**2).mean():.1f}×")

print(f"\nAttention distribution (KL divergence from reference):")
eps   = 1e-8
probs_ref   = sim['probs_ref']
probs_naive = sim['probs_naive']
probs_turbo = sim['probs_turbo']
kl_naive = (probs_ref * (probs_ref / (probs_naive + eps) + eps).log()).sum().item()
kl_turbo = (probs_ref * (probs_ref / (probs_turbo + eps) + eps).log()).sum().item()
print(f"  Naive KL divergence: {kl_naive:.5f}")
print(f"  TurboQuant KL:       {kl_turbo:.5f}")

In [ ]:
# 📊 Visualize attention weight distortion
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

token_idx = range(128)

# --- Top: attention probability distributions ---
axes[0, 0].plot(token_idx, sim['probs_ref'].numpy(),
                color='black',   linewidth=2,   label='Reference (no compression)')
axes[0, 0].plot(token_idx, sim['probs_naive'].numpy(),
                color='crimson', linewidth=1.5, label='Naive quant', alpha=0.8, linestyle='--')
axes[0, 0].plot(token_idx, sim['probs_turbo'].numpy(),
                color='seagreen',linewidth=1.5, label='TurboQuant',  alpha=0.8, linestyle='-.')
axes[0, 0].set_title('Attention Weights Over Sequence', fontweight='bold')
axes[0, 0].set_xlabel('Token position')
axes[0, 0].set_ylabel('Attention probability')
axes[0, 0].legend(fontsize=8)

# --- Top right: absolute error in attention weights ---
axes[0, 1].plot(token_idx,
                (sim['probs_ref'] - sim['probs_naive']).abs().numpy(),
                color='crimson', linewidth=1.5, label='Naive error')
axes[0, 1].plot(token_idx,
                (sim['probs_ref'] - sim['probs_turbo']).abs().numpy(),
                color='seagreen', linewidth=1.5, label='TurboQuant error')
axes[0, 1].set_title('|Attention Weight Error| per Token', fontweight='bold')
axes[0, 1].set_xlabel('Token position')
axes[0, 1].set_ylabel('Absolute error')
axes[0, 1].legend(fontsize=8)

# --- Bottom left: key vector norms (shows where outliers are) ---
key_norms = sim['keys'].norm(dim=-1).numpy()
axes[1, 0].bar(token_idx, key_norms, color='steelblue', alpha=0.7, width=1.0)
axes[1, 0].axhline(np.percentile(key_norms, 95), color='red', linestyle='--',
                    linewidth=1.5, label='95th percentile')
axes[1, 0].set_title('Key Vector Norms\n(spikes = tokens with outliers)', fontweight='bold')
axes[1, 0].set_xlabel('Token position')
axes[1, 0].set_ylabel('||key||')
axes[1, 0].legend(fontsize=8)

# --- Bottom right: attention output error per dimension ---
ref_vec   = sim['attn_ref'].numpy()
naive_vec = sim['attn_naive'].numpy()
turbo_vec = sim['attn_turbo'].numpy()
x_d = range(len(ref_vec))
axes[1, 1].bar(x_d, np.abs(ref_vec - naive_vec), color='crimson',  alpha=0.7,
               label='Naive error', width=0.8)
axes[1, 1].bar(x_d, np.abs(ref_vec - turbo_vec), color='seagreen', alpha=0.7,
               label='TurboQuant error', width=0.8)
axes[1, 1].set_title('Attention Output Error per Dimension', fontweight='bold')
axes[1, 1].set_xlabel('Output dimension')
axes[1, 1].set_ylabel('|error|')
axes[1, 1].legend(fontsize=8)

plt.suptitle('🔍 KV Cache Compression: Impact on Attention Quality',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 🎯 Final Output — The Complete TurboQuant Dashboard

Let us produce a beautiful, comprehensive visualization that brings together everything we have learned — worthy of a screenshot!

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# The TurboQuant Complete Dashboard
# ═══════════════════════════════════════════════════════════════════════════
torch.manual_seed(42)
np.random.seed(42)

dim      = 64
n_tokens = 300
bits     = 4
tq       = TurboQuant(dim=dim, bits=bits)

# Generate data with mixed outlier intensities
keys = torch.randn(n_tokens, dim) * 0.4
n_outlier = 15
outlier_tokens = torch.randperm(n_tokens)[:n_outlier]
outlier_dims   = torch.randperm(dim)[:3]
for t in outlier_tokens:
    mag = torch.rand(1).item() * 8 + 3
    sgn = 1 if torch.rand(1) > 0.5 else -1
    keys[t, outlier_dims[torch.randint(0, 3, (1,)).item()]] = mag * sgn

keys_naive = naive_quantize(keys, bits=bits)
compressed = tq.compress(keys)
keys_turbo = tq.decompress(compressed)

err_naive = (keys - keys_naive).pow(2).mean(dim=-1).numpy()
err_turbo = (keys - keys_turbo).pow(2).mean(dim=-1).numpy()
key_norms = keys.norm(dim=-1).numpy()

# ── The big figure ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 13), facecolor='#0d1117')
fig.patch.set_facecolor('#0d1117')

def dark_ax(ax, title=''):
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9', labelsize=8)
    ax.xaxis.label.set_color('#c9d1d9')
    ax.yaxis.label.set_color('#c9d1d9')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    if title:
        ax.set_title(title, color='#e6edf3', fontweight='bold', fontsize=10, pad=8)
    return ax

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.38,
                        left=0.06, right=0.97, top=0.88, bottom=0.07)

# ── Panel 1: A single key vector before/after rotation ──────────────────────
ax1 = dark_ax(fig.add_subplot(gs[0, :2]), 'Panel 1: Key Vector — Original vs. Rotated')
H   = hadamard_matrix(dim)
sample_key     = keys[outlier_tokens[0]].numpy()
sample_key_rot = (H @ keys[outlier_tokens[0]]).numpy()
x = np.arange(dim)
ax1.bar(x, sample_key,     color='#58a6ff', alpha=0.8, width=0.9, label='Original')
ax1.bar(x, sample_key_rot, color='#bc8cff', alpha=0.6, width=0.9, label='After H rotation')
ax1.axhline(0, color='#c9d1d9', linewidth=0.5)
ax1.set_xlabel('Dimension'); ax1.set_ylabel('Value')
ax1.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)

# ── Panel 2: MSE per token ───────────────────────────────────────────────────
ax2 = dark_ax(fig.add_subplot(gs[0, 2:]), 'Panel 2: Per-Token Reconstruction MSE')
ax2.plot(err_naive, color='#ff7b72', linewidth=1.5, label='Naive', alpha=0.9)
ax2.plot(err_turbo, color='#3fb950', linewidth=1.5, label='TurboQuant', alpha=0.9)
ax2.fill_between(range(n_tokens), err_naive, err_turbo, alpha=0.15, color='#58a6ff')
ax2.set_xlabel('Token index'); ax2.set_ylabel('MSE')
ax2.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)

# ── Panel 3: MSE vs. outlier magnitude (sweep) ───────────────────────────────
ax3 = dark_ax(fig.add_subplot(gs[1, :2]),
               'Panel 3: MSE vs. Outlier Magnitude (4-bit)')
mags = np.linspace(0, 14, 35)
msn, mst = [], []
tq_sweep = TurboQuant(dim=32, bits=4)
for m in mags:
    torch.manual_seed(0)
    v = torch.randn(100, 32) * 0.3
    v[:, 5] = m * (torch.randint(0,2,(100,))*2-1).float()
    msn.append(((v - naive_quantize(v, 4))**2).mean().item())
    mst.append(((v - tq_sweep.decompress(tq_sweep.compress(v)))**2).mean().item())
ax3.plot(mags, msn, color='#ff7b72', linewidth=2.5, label='Naive')
ax3.plot(mags, mst, color='#3fb950', linewidth=2.5, label='TurboQuant')
ax3.fill_between(mags, msn, mst, alpha=0.2, color='#58a6ff')
ax3.set_xlabel('Outlier magnitude'); ax3.set_ylabel('MSE')
ax3.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)

# ── Panel 4: Improvement ratio heatmap (bits × outlier_fraction) ─────────────
ax4 = dark_ax(fig.add_subplot(gs[1, 2:]),
               'Panel 4: Improvement Ratio Heatmap\n(TurboQuant MSE / Naive MSE)')
bits_range  = [2, 4, 8]
outlier_mag = [0, 2, 4, 6, 8, 10]
ratio_grid  = np.zeros((len(bits_range), len(outlier_mag)))
for bi, b in enumerate(bits_range):
    tq_h = TurboQuant(dim=32, bits=b)
    for oi, om in enumerate(outlier_mag):
        torch.manual_seed(0)
        v = torch.randn(100, 32) * 0.3
        v[:, 5] = om * (torch.randint(0,2,(100,))*2-1).float()
        mn = ((v - naive_quantize(v, b))**2).mean().item()
        mt = ((v - tq_h.decompress(tq_h.compress(v)))**2).mean().item()
        ratio_grid[bi, oi] = mn / (mt + 1e-9)

import matplotlib.colors as mcolors
cmap = plt.cm.RdYlGn
norm = mcolors.LogNorm(vmin=0.5, vmax=ratio_grid.max())
im   = ax4.imshow(ratio_grid, aspect='auto', cmap=cmap, norm=norm)
ax4.set_xticks(range(len(outlier_mag))); ax4.set_xticklabels(outlier_mag)
ax4.set_yticks(range(len(bits_range)));  ax4.set_yticklabels([f'{b}b' for b in bits_range])
ax4.set_xlabel('Outlier magnitude'); ax4.set_ylabel('Bits')
for bi in range(len(bits_range)):
    for oi in range(len(outlier_mag)):
        v_text = ratio_grid[bi, oi]
        ax4.text(oi, bi, f'{v_text:.1f}×', ha='center', va='center',
                 fontsize=8, color='black' if v_text < 50 else 'white', fontweight='bold')
cb = fig.colorbar(im, ax=ax4, shrink=0.85)
cb.ax.tick_params(colors='#c9d1d9', labelsize=7)
cb.set_label('Improvement ×', color='#c9d1d9', fontsize=8)

# ── Panel 5: Summary bar chart ───────────────────────────────────────────────
ax5 = dark_ax(fig.add_subplot(gs[2, :2]), 'Panel 5: 4-bit Summary Statistics')
categories  = ['MSE', 'Max Error', '1 - CosSim']
mean_key_mse_n = err_naive.mean()
mean_key_mse_t = err_turbo.mean()
max_naive  = (keys - keys_naive).abs().max().item()
max_turbo  = (keys - keys_turbo).abs().max().item()
import torch.nn.functional as F
cs_naive = 1 - F.cosine_similarity(keys, keys_naive, dim=-1).mean().item()
cs_turbo = 1 - F.cosine_similarity(keys, keys_turbo, dim=-1).mean().item()

naive_stats = [mean_key_mse_n, max_naive, cs_naive]
turbo_stats = [mean_key_mse_t, max_turbo, cs_turbo]

xb = np.arange(3)
ax5.bar(xb - 0.2, naive_stats, 0.4, color='#ff7b72', label='Naive', alpha=0.9)
ax5.bar(xb + 0.2, turbo_stats, 0.4, color='#3fb950', label='TurboQuant', alpha=0.9)
ax5.set_xticks(xb); ax5.set_xticklabels(categories)
ax5.set_ylabel('Error metric value')
ax5.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)
for i, (n, t) in enumerate(zip(naive_stats, turbo_stats)):
    if t > 1e-6:
        ax5.text(i + 0.2, t + max(naive_stats)*0.02, f'{n/t:.1f}×',
                 ha='center', fontsize=8, color='#f0e68c', fontweight='bold')

# ── Panel 6: Memory savings diagram ─────────────────────────────────────────
ax6 = dark_ax(fig.add_subplot(gs[2, 2:]), 'Panel 6: Memory Budget for KV Cache')
labels_mem = ['FP16\n(baseline)', 'INT8\n(no rotation)', 'INT4\nNaive', 'INT4\nTurboQuant']
memory_mb  = [100, 50, 25, 25]
quality    = [1.00, 0.92, 0.61, 0.94]     # relative quality (illustrative)
colors_bar = ['#8b949e', '#d29922', '#ff7b72', '#3fb950']

bars = ax6.bar(labels_mem, memory_mb, color=colors_bar, alpha=0.9, width=0.55)
ax6.set_ylabel('Relative Memory (100 = FP16 baseline)')
ax6.set_ylim(0, 120)
for bar, mem, qual in zip(bars, memory_mb, quality):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{mem}%\nQ={qual:.2f}', ha='center', va='bottom',
             color='#e6edf3', fontsize=8, fontweight='bold')
ax6.axhline(25, color='#58a6ff', linestyle='--', linewidth=1.5, alpha=0.6)
ax6.text(3.45, 27, '4× reduction', color='#58a6ff', fontsize=8)

# ── Main title ────────────────────────────────────────────────────────────────
fig.text(0.5, 0.96,
         '🚀 TurboQuant: Rotating Your Way to Better Cache Compression',
         ha='center', va='center', fontsize=16, fontweight='bold',
         color='#e6edf3',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1f6feb',
                   edgecolor='#388bfd', alpha=0.9))
fig.text(0.5, 0.925,
         f'dim={dim} | 4-bit | {n_tokens} tokens | '
         f'Avg improvement: {err_naive.mean()/err_turbo.mean():.1f}×',
         ha='center', va='center', fontsize=10, color='#8b949e')

plt.savefig('turboQuant_dashboard.png', dpi=180, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("\n✅ Dashboard saved to turboQuant_dashboard.png")
print(f"📊 Average MSE improvement: {err_naive.mean()/err_turbo.mean():.1f}×")
print(f"📊 Max error improvement:   {(keys-keys_naive).abs().max()/(keys-keys_turbo).abs().max():.1f}×")
print(f"📊 Memory reduction:        4× (16-bit → 4-bit)")

## 9. Reflection and Next Steps

### 🤔 Reflection Questions

Let us consolidate what we learned by thinking through these questions. Work through each one before reading the answer hint.

**Q1: Why does the Hadamard transform specifically, and not just any random rotation?**

*Hint: Think about computational cost. A dense random rotation matrix $R \in \mathbb{R}^{d \times d}$ requires $O(d^2)$ operations to apply. The Hadamard transform can be computed in $O(d \log d)$ using the fast butterfly algorithm — and it only uses additions and subtractions, no multiplications (the $\pm 1/\sqrt{d}$ structure means no costly floating-point multiplies per element).*

**Q2: When we rotate → quantize → rotate back, why is the total information loss not zero?**

*Hint: The rotation itself is lossless ($R^{-1}$ exists exactly). The lossy step is quantization, which is performed in the rotated space. Rotating back does not undo quantization — it just transforms the quantization error from the rotated space back to the original space. The key insight is that we get smaller quantization errors because the rotated vector has no outliers, so the 16-level grid fits all values well.*

**Q3: Would TurboQuant help if all dimensions had the same magnitude (no outliers)?**

*Hint: Theoretically, no — or at least very little. Uniform quantization already works well when all values are similar in magnitude. TurboQuant's benefit is specifically proportional to the "peakedness" of the distribution. If the original vector is already flat, rotation does not help much. This is why the heatmap in Panel 4 shows small improvements at outlier magnitude = 0.*

**Q4: In a real transformer, where exactly are H and H.T applied to avoid the per-token cost of rotating?**

*Hint: The key observation from the QuaRot paper (Ashkboos et al., 2024) is that you can absorb the Hadamard rotation directly into the weight matrices of the linear layers that produce keys and values. Instead of computing $\mathbf{k} = W_K \mathbf{x}$ and then $H\mathbf{k}$, you pre-compute $\tilde{W}_K = H W_K$ once offline. Then at inference, you compute $H\mathbf{k} = \tilde{W}_K \mathbf{x}$ for free — the rotation is fused into the matmul that was happening anyway!*

### 🏆 Optional Challenges

Ready to go deeper? Here are three challenges of increasing difficulty:

---

**Challenge 1: Per-Channel Scaling (⭐⭐)**

Our implementation uses one scale per token (per row). In practice, "per-channel" quantization uses a separate scale per dimension. Implement `TurboQuantPerChannel` where:
- After rotation, compute a separate scale per *column* (feature dimension) rather than per row
- Measure whether per-channel scaling gives better results than per-token scaling
- When would per-channel be better? When would per-token be better?

---

**Challenge 2: Absorbing Rotation into Weight Matrices (⭐⭐⭐)**

This is the core insight from QuaRot. Suppose we have a simplified key computation:

In [ ]:
W_K = torch.randn(64, 64)
x   = torch.randn(64)
k   = W_K @ x           # original key
k_rotated = H @ k       # TurboQuant needs this

Show that `k_rotated = (H @ W_K) @ x`. Pre-compute `W_K_fused = H @ W_K` and verify that `W_K_fused @ x` equals `H @ (W_K @ x)`. Measure the wall-clock time savings for a large batch of tokens.

---

**Challenge 3: Mixed-Precision TurboQuant (⭐⭐⭐⭐)**

Not all tokens need the same bit width. Tokens with high-norm keys (outlier-prone) might deserve 8 bits, while quiet tokens can get 2 bits — saving memory while maintaining quality. Implement a `MixedPrecisionTurboQuant` class that:
- Computes a "difficulty score" for each token (e.g., norm of the key vector)
- Assigns bit width based on difficulty (2-bit for easy tokens, 4-bit for medium, 8-bit for hard)
- Measures the average effective bits per weight and the resulting MSE
- Plots the Pareto frontier of memory vs. quality

---

### 📚 What to Read Next

- **QuaRot** (Ashkboos et al., 2024): "QuaRot: Outlier-Free 4-Bit Inference in Rotated LLMs" — the paper this notebook is based on. Shows how to absorb rotations into weight matrices.

- **LLM.int8()** (Dettmers et al., 2022): The paper that first systematically documented outlier dimensions in transformers and proposed mixed-precision quantization to handle them.

- **SmoothQuant** (Xiao et al., 2022): A complementary approach — instead of rotating, multiply the activations by a per-channel scale factor to "smooth" outliers before quantization.

- **KVQuant** (Hooper et al., 2024): Combines per-channel quantization with non-uniform (codebook-based) quantization grids for even better KV cache compression.

In [ ]:
# 🎉 Final summary cell — print your achievement!
print("=" * 60)
print("🚀 TURBOQUANT NOTEBOOK COMPLETE!")
print("=" * 60)
print()
print("What you built from scratch today:")
print("  ✅ Uniform quantization with full understanding of the math")
print("  ✅ Hadamard matrix construction (recursive) + FWHT (O(n log n))")
print("  ✅ The full TurboQuant pipeline (rotate → quant → store → load → dequant → unrotate)")
print("  ✅ Ablation study across bit widths and outlier intensities")
print("  ✅ KV cache simulation with attention quality measurement")
print("  ✅ Publication-quality dashboard visualization")
print()
print("Key numbers to remember:")
print(f"  📉  Memory: 16-bit → 4-bit = 4× reduction")
print(f"  📈  Quality: TurboQuant recovers most of that quality back")
print(f"  ⚡  Speed: FWHT is O(d log d) — negligible overhead")
print(f"  🔑  Trick: absorb H into weight matrices for zero inference cost")
print()
print("The big idea in one sentence:")
print("  Rotation spreads outliers uniformly, making 4-bit quantization")
print("  work as well as 8-bit — for free, if you're clever about it.")
print()
print("See you in Section 7! 🎓")